## Excipient-Aware Drug Retrieval Using Medaka




In [ ]:
import os
from pathlib import Path
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt

# Configurable input path.
MEDAKA_CSV = Path(os.environ.get("MEDAKA_CSV", "../data/medaka_v1.csv"))

OUT_DIR = Path("medaka_use_case_outputs")
OUT_DIR.mkdir(exist_ok=True)

kg = pd.read_csv(MEDAKA_CSV)
kg.columns = [c.strip().lower() for c in kg.columns]
kg = kg.rename(columns={"subject": "drug", "relation": "relation", "object": "object"})

required_cols = {"drug", "relation", "object"}
missing = required_cols - set(kg.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in ["drug", "relation", "object"]:
    kg[col] = kg[col].astype(str).str.strip().str.lower()

kg = kg[kg["relation"].isin(["hasactiveingredient", "hasinactiveingredient"])].copy()

print(f"Loaded {len(kg):,} ingredient triples from {MEDAKA_CSV}")
kg.head()


In [ ]:
# Build Ingredient maps

active_map = defaultdict(set)
inactive_map = defaultdict(set)

for drug, relation, obj in kg[["drug", "relation", "object"]].itertuples(index=False):
    if relation == "hasactiveingredient":
        active_map[drug].add(obj)
    elif relation == "hasinactiveingredient":
        inactive_map[drug].add(obj)

active_groups = defaultdict(set)
for drug, ingredients in active_map.items():
    if ingredients:
        active_groups[frozenset(ingredients)].add(drug)

print(f"Products with active ingredients: {len(active_map):,}")
print(f"Products with inactive ingredients: {len(inactive_map):,}")
print(f"Active-ingredient groups: {len(active_groups):,}")


In [ ]:
EXCIPIENTS = {
    "lactose": ["lactose"],
    "sodium": ["sodium"],
    "ethanol/alcohol": ["ethanol", "alcohol"],
    "propylene glycol": ["propylene glycol"],
    "sucrose": ["sucrose"],
    "sorbitol": ["sorbitol"],
    "aspartame": ["aspartame"],
    "benzyl alcohol": ["benzyl alcohol"],
    "soya/soy oil": ["soya", "soy"],
    "peanut/arachis oil": ["peanut", "arachis"],
    "azo dyes": ["tartrazine", "sunset yellow", "azorubine", "carmoisine", "ponceau"],
}

def has_term(values, terms):
    values = [str(v).lower() for v in values if pd.notna(v)]
    return any(term in value for value in values for term in terms)

rows = []
examples = []

for excipient, terms in EXCIPIENTS.items():
    flagged = {
        drug for drug, ingredients in inactive_map.items()
        if has_term(ingredients, terms)
    }

    useful_groups = 0
    flagged_with_alt = set()
    clean_candidates = set()

    for active_sig, drugs in active_groups.items():
        if len(drugs) < 2:
            continue

        flagged_in_group = drugs & flagged
        clean_in_group = drugs - flagged

        if flagged_in_group and clean_in_group:
            useful_groups += 1
            flagged_with_alt.update(flagged_in_group)
            clean_candidates.update(clean_in_group)

            examples.append({
                "excipient": excipient,
                "active_ingredients": "; ".join(sorted(str(x) for x in active_sig if pd.notna(x))),
                "flagged_product_example": sorted(flagged_in_group)[0],
                "candidate_alternatives": "; ".join(sorted(clean_in_group)[:5]),
                "n_flagged_products": len(flagged_in_group),
                "n_clean_candidates": len(clean_in_group),
            })

    rows.append({
        "excipient": excipient,
        "products_containing_excipient": len(flagged),
        "products_with_same_active_clean_alt": len(flagged_with_alt),
        "active_ingredient_groups_with_clean_alt": useful_groups,
        "unique_clean_candidate_alternatives": len(clean_candidates),
    })

summary = pd.DataFrame(rows).sort_values("products_containing_excipient", ascending=False)
examples = pd.DataFrame(examples)

summary.to_csv(OUT_DIR / "excipient_summary.csv", index=False)
examples.to_csv(OUT_DIR / "excipient_candidate_examples.csv", index=False)

summary


In [ ]:
# Selected candidate examples

selected_excipients = [
    "lactose",
    "aspartame",
    "benzyl alcohol",
    "propylene glycol",
    "soya/soy oil",
]

selected = examples[examples["excipient"].isin(selected_excipients)].copy()

selected = (
    selected
    .sort_values(["excipient", "n_clean_candidates"], ascending=[True, False])
    .groupby("excipient")
    .head(3)
)

selected.to_csv(OUT_DIR / "selected_substitution_examples.csv", index=False)
selected


In [ ]:
# generate figure

FIGURE_EXCIPIENTS = [
    "Lactose",
    "Ethanol/alcohol",
    "Propylene glycol",
    "Sucrose",
    "Sorbitol",
    "Aspartame",
    "Benzyl alcohol",
    "Soya/soy oil",
]

fig_df = summary[summary["excipient"].isin(FIGURE_EXCIPIENTS)].copy()
fig_df = fig_df.sort_values("products_containing_excipient", ascending=True)

total_color = "#c6dbef"
actionable_color = "#80b2d3"

fig, ax = plt.subplots(figsize=(7.0, 4.6))

ax.barh(
    fig_df["excipient"],
    fig_df["products_containing_excipient"],
    color=total_color,
    edgecolor="none",
    label="Drugs containing excipient",
)

ax.barh(
    fig_df["excipient"],
    fig_df["products_with_same_active_clean_alt"],
    color=actionable_color,
    edgecolor="none",
    label="Drugs with an excipient-free alternative",
)

for _, row in fig_df.iterrows():
    total = row["products_containing_excipient"]
    actionable = row["products_with_same_active_clean_alt"]
    pct = 100 * actionable / total if total else 0

    ax.text(
        total + 35,
        row["excipient"],
        f"{pct:.0f}%",
        va="center",
        ha="left",
        fontsize=9,
    )


ax.set_xlabel("Number of products")
ax.set_ylabel("")

ax.set_xlim(0, 1600)
ax.set_xticks([0, 350, 700, 1050, 1400])
ax.minorticks_off()
ax.tick_params(axis="x", which="both", length=0)
ax.tick_params(axis="y", which="both", length=0)


ax.grid(False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(0.98, 0.08))


plt.tight_layout()

fig_path = OUT_DIR / "medaka_excipient_substitution_funnel_clean.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=300)
plt.show()

print(f"Saved figure to: {fig_path}")
